# Customer and subscription EDA

**Objective.** Establish the account and contractual episode grains, assess lifecycle coverage and test whether an account can safely stand in for an episode in later analysis.

**Sources.** Fieldwise Cloud is the fictional seed-42 GB SaaS reference for April 2021 to March 2024. The supported generator reconstructs it in an isolated temporary directory. A fresh local Registry assigns new canonical UUIDs on each reconstruction, so aggregates are compared and identifiers stay out of outputs. Misata is an independently acquired fictional SaaS source covering ten country markets, including the UK. Its source-native tables are analysed separately.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython import get_ipython
from IPython.display import display

# Resolve the repository before importing the local src-layout package.
project = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "src" / "nowa_customer_intelligence").is_dir()
    ),
    None,
)
if project is None:
    raise RuntimeError(
        "Start the Jupyter kernel from the Nowa Customer Intelligence repository "
        "root or one of its subdirectories."
    )

source_dir = str(project / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

get_ipython().run_line_magic("matplotlib", "inline")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 4), "axes.titleweight": "bold"})

## Fieldwise Cloud population and contract quality

In [ ]:
from nowa_customer_intelligence.data.eda_inputs import REFERENCE_CUTOFF, fieldwise_reference
from nowa_customer_intelligence.data.synthetic.lifecycle import LifecycleKind

with fieldwise_reference() as (run, _export):
    population = run.population
    quality = {
        table.dataset: {
            "rows": len(table.rows),
            "exact duplicate rows": len(table.rows) - len(set(table.rows)),
        }
        for table in population.tables
    }
    missing = {
        table.dataset: {
            column: sum(row[index] == "" for row in table.rows)
            for index, column in enumerate(table.columns)
        }
        for table in population.tables
    }

episodes = pd.DataFrame(
    [
        {
            "episode": row.episode.subscription_id,
            "account": row.episode.customer_id,
            "started": row.episode.start_date,
            "ended": row.episode.end_date,
            "status": row.episode.subscription_status.value,
            "plan": row.episode.plan_id,
            "frequency": row.episode.billing_frequency,
            "prior": row.prior_subscription_id,
            "termination": next(
                (fact.effective_at for fact in row.facts if fact.kind is LifecycleKind.TERMINATED),
                None,
            ),
        }
        for row in population.episodes
    ]
)
for field in ("started", "ended"):
    episodes[field] = pd.to_datetime(episodes[field])
accounts = {row.customer_id for row in population.customers}
episode_ids = set(episodes["episode"])
references = pd.Series(
    {
        "episodes with unknown account": (~episodes["account"].isin(accounts)).sum(),
        "usage rows with unknown episode": sum(
            r.subscription_id not in episode_ids for r in population.usage
        ),
        "snapshots with unknown episode": sum(
            r.subscription_id not in episode_ids for r in population.snapshots
        ),
        "invoices with unknown episode": sum(
            c.invoice.subscription_id not in episode_ids for c in population.charges
        ),
    },
    name="unresolved references",
)
account_sizes = episodes.groupby("account").size().value_counts().sort_index()
account_sizes.index.name = "episodes per account"
display(pd.DataFrame(quality).T, references, account_sizes.rename("accounts"))
display(
    pd.DataFrame(missing).T.stack().loc[lambda s: s > 0].astype(int).rename("blank source fields")
)
display(
    pd.Series(
        {
            "first customer creation": min(row.created_at for row in population.customers),
            "last customer creation": max(row.created_at for row in population.customers),
            "first episode establishment": episodes["started"].min().date(),
            "last episode establishment": episodes["started"].max().date(),
        },
        name="observed dates",
    )
)
display(pd.crosstab(episodes["plan"], episodes["frequency"]))
assert references.sum() == 0

## Lifecycle, cohorts and observed duration

In [ ]:
indexed = episodes.set_index("episode", verify_integrity=True)
reactivated = episodes.loc[episodes["prior"].notna()].copy()
predecessors = indexed.reindex(reactivated["prior"]).reset_index(drop=True)
separation = (reactivated["started"].reset_index(drop=True) - predecessors["ended"]).dt.days
assert (separation > 0).all()
assert predecessors["account"].tolist() == reactivated["account"].tolist()
assert reactivated["episode"].ne(reactivated["prior"]).all()
episodes["observed_days"] = (
    episodes["ended"].fillna(pd.Timestamp(REFERENCE_CUTOFF.date())) - episodes["started"]
).dt.days
episodes["episode_group"] = (
    episodes["termination"]
    .notna()
    .map({True: "Confirmed termination", False: "Observed to cutoff"})
)
established = episodes.groupby(episodes["started"].dt.to_period("Q")).size()
reactivation = reactivated.groupby(reactivated["started"].dt.to_period("Q")).size()
cohorts = (
    pd.concat(
        [established.rename("All establishments"), reactivation.rename("Reactivations")], axis=1
    )
    .fillna(0)
    .astype(int)
)
display(episodes.groupby("episode_group")["observed_days"].describe().round(1))
display(separation.describe().round(1).rename("Days between predecessor end and reactivation"))
display(episodes["status"].value_counts().rename("episodes"))
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4), layout="constrained")
cohorts.plot.bar(ax=left, color=["#23667d", "#db8351"])
left.set(
    title="Fieldwise episode establishment cohorts", xlabel="Calendar quarter", ylabel="Episodes"
)
episodes.boxplot(column="observed_days", by="episode_group", ax=right, grid=False)
right.set(title="Fieldwise observed episode duration", xlabel="End evidence", ylabel="Days")
fig.suptitle("")
plt.show()
print(
    f"Fieldwise: {len(accounts):,} accounts, {len(episodes):,} episodes, "
    f"{episodes['termination'].notna().sum():,} confirmed terminations, "
    f"{len(reactivated):,} reactivations."
)

## Complementary Misata account and billing analysis

In [ ]:
from nowa_customer_intelligence.data.public_sources.acquisition import verify_acquisition
from nowa_customer_intelligence.data.public_sources.misata import load_misata

source = project / "data" / "external" / "misata_saas"
verify_acquisition(source)
misata = load_misata(source)
misata_accounts = pd.DataFrame(misata.tables["accounts"])
misata_episodes = pd.DataFrame(misata.tables["subscriptions"])
misata_invoices = pd.DataFrame(misata.tables["invoices"])
status_by_plan = pd.crosstab(
    misata_episodes["account_id"].map(misata_accounts.set_index("account_id")["plan"]),
    misata_episodes["status"],
).rename_axis("plan")
invoice_coverage = misata_invoices.groupby("account_id").size()
display(pd.DataFrame(misata.quality).T[["rows", "primary_key_duplicates", "orphaned_accounts"]])
display(status_by_plan)
display(
    pd.Series(
        {
            "accounts with invoices": invoice_coverage.size,
            "accounts without invoices": len(misata_accounts) - invoice_coverage.size,
            "first invoice (UTC)": misata_invoices["invoice_date"].min(),
            "last invoice (UTC)": misata_invoices["invoice_date"].max(),
        },
        name="Misata invoice coverage",
    )
)
ax = status_by_plan.plot.bar(stacked=True, color=["#23667d", "#db8351", "#879865", "#8979ab"])
ax.set(
    title="Misata source subscription status by plan", xlabel="Source plan", ylabel="Subscriptions"
)
ax.tick_params(axis="x", labelrotation=0)
plt.show()

## Interpretation and conclusion

Fieldwise has 800 accounts with one episode and 200 with two. Each reactivation has a distinct canonical episode ID, the same account owner and a predecessor that ended exactly one day earlier. This fixed interval and the 200 successor episodes are properties of the generator scenario, and provide no estimate of GB reactivation timing or rates. The date summaries describe **observed** lifetimes. Episodes open at the extraction cutoff have incomplete lifetimes and should not be pooled with completed durations as if both were fully observed. Blank lifecycle or payment dates may be structurally expected. Blank optional demographics or usage metrics constrain what can safely be analysed.

Misata supplies one source subscription per account and account-level invoice references. A source status or identifier does not establish a Nowa canonical episode, authoritative termination history or historical feature availability. Billing coverage describes the acquired extract only. The two fictional populations remain separate and their distributions are not empirical estimates of the GB market.

For later phases, retain account and episode as separate grouping keys, preserve event evidence and availability, and confirm which optional fields are comparable across observation periods before deriving predictors.